In [ ]:
import cftime
import pandas as pd
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
from scipy.ndimage import label
from pathlib import Path
import pyarrow as pa
from collections import Counter
import regionmask
import matplotlib.patheffects as path_effects 
import cartopy.io.shapereader as shpreader
import seaborn as sns
from scipy import stats

# Ticks and Plot Features

In [ ]:
import numpy as np

def generate_custom_ticks_1(grid_values):
    # 1. Get exact data limits
    exact_min = np.nanmin(grid_values)
    exact_max = np.nanmax(grid_values)
    
    if np.isnan(exact_min): 
        exact_min, exact_max = -7.0, 7.0
        
    # 2. Find the largest absolute value to force symmetry around 0
    max_abs = int(np.ceil(max(abs(exact_min), abs(exact_max))))
    
    # Force a minimum symmetric boundary (e.g., if max change is only 1, force a clean map)
    if max_abs == 0: 
        max_abs = 1
        
    # 3. Establish symmetric exact visual boundaries for pcolormesh (vmin / vmax)
    exact_min_sym = -max_abs
    exact_max_sym = max_abs
    
    # 4. Generate whole-number ticks from -max_abs to +max_abs
    # We step by 1 or 2 depending on how wide the span is to keep the colorbar clean
    step = 1
    custom_ticks = list(range(-max_abs, max_abs + 1, step))
    
    # 5. Create clean string labels without decimal points
    tick_labels = [f"{t}" for t in custom_ticks]
    
    return custom_ticks, tick_labels, exact_max_sym, exact_min_sym

In [ ]:
import numpy as np

def generate_custom_ticks_05(grid_values):
    # 1. Get exact data limits
    exact_min = np.nanmin(grid_values)
    exact_max = np.nanmax(grid_values)
    
    if np.isnan(exact_min): exact_min, exact_max = 0.0, 2.0
    
    # 2. Round outward to the nearest 0.5 step to establish the uniform grid
    grid_start = np.floor(exact_min * 2) / 2
    grid_end = np.ceil(exact_max * 2) / 2
    
    # 3. Generate the full uniform step array (inclusive of grid_end)
    full_grid = np.arange(grid_start, grid_end + 0.1, 0.5)
    
    # 4. Slice off the fake outer boundaries to leave only the true middle ticks
    middle_ticks = list(full_grid[1:-1])
    
    # 5. Group together: [Exact Min, Middle Ticks..., Exact Max]
    custom_ticks = middle_ticks
    
    # 6. Generate text labels matching the spacing rules
    tick_labels = []
    for t in middle_ticks:
        if t % 1 == 0:
            tick_labels.append(f"{t:.1f}") # Pure whole numbers get no decimals
        else:
            tick_labels.append(f"{t:.1f}")  # Half-steps get 1 decimal place (e.g., 2.5)
            
    return custom_ticks, tick_labels, exact_max, exact_min

In [ ]:
import numpy as np

def generate_custom_ticks_02(grid_values):
    # 1. Get exact data limits
    exact_max = np.nanmax(grid_values)
    
    # 2. Round outward to the nearest 0.5 step to establish the uniform grid
    grid_start = np.floor(-exact_max * 2) / 2
    grid_end = np.ceil(exact_max * 2) / 2
    
    # 3. Generate the full uniform step array (inclusive of grid_end)
    full_grid = np.arange(grid_start, grid_end + 0.1, .1)
    
    # 4. Slice off the fake outer boundaries to leave only the true middle ticks
    middle_ticks = list(full_grid[1:-1])
    
    # 5. Group together: [Exact Min, Middle Ticks..., Exact Max]
    custom_ticks = [-exact_max] + middle_ticks + [exact_max]
    
    # 6. Generate text labels matching the spacing rules
    tick_labels = []
    for t in custom_ticks:
        if t % 1 == 0:
            tick_labels.append(f"{t:.1f}") # Pure whole numbers get no decimals
        else:
            tick_labels.append(f"{t:.1f}")  # Half-steps get 1 decimal place (e.g., 2.5)
            
    return custom_ticks, tick_labels, exact_max, -exact_max

In [ ]:

# --- STEP 1: Load Canada and Mexico Geometries once ---
shpfilename = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_0_countries')
reader = shpreader.Reader(shpfilename)
records = reader.records()

canada_geom = None
mexico_geom = None

for record in records:
    country_name = record.attributes.get('NAME')
    if country_name == 'Canada':
        canada_geom = record.geometry
    elif country_name == 'Mexico':
        mexico_geom = record.geometry

# Load 50m Lakes Shapefile from Natural Earth
lakes_shp = shpreader.natural_earth(resolution='50m', category='physical', name='lakes')
lakes_reader = shpreader.Reader(lakes_shp)

# Names of the Great Lakes to isolate
great_lakes_names = {'Lake Superior', 'Lake Michigan', 'Lake Huron', 'Lake Erie', 'Lake Ontario'}

# Filter out only the Great Lakes geometries
great_lakes_geoms = []
for record in lakes_reader.records():
    # 'name' is the attribute key for the lake's name in Natural Earth
    lake_name = record.attributes.get('name')
    if lake_name in great_lakes_names:
        great_lakes_geoms.append(record.geometry)

# Use Natural Earth's defined regions for US States (50m resolution)
us_states = regionmask.defined_regions.natural_earth_v5_0_0.us_states_50

# HIST/FTR Thirstwave File

In [ ]:
# Define file paths
hist = '/data1/michsh/CSV/HIST_derived_metrics_2.csv'
ftr = '/data1/michsh/CSV/FUT_derived_metrics_2.csv'

# Frequency

In [ ]:
frq_chunks_hist = []
frq_chunks_ftr = []

# Load annual freqeuncy for each member, lat, lon, and year.
for frq_chunk_hist in pd.read_csv(hist, chunksize=500000):
    grouped_frq_chunk_hist = (
        frq_chunk_hist
        .groupby(['AMOC', 'member', 'lat', 'lon', 'year'])['annual_frequency']
        .first()
        .reset_index()
    )
    frq_chunks_hist.append(grouped_frq_chunk_hist)

for frq_chunk_ftr in pd.read_csv(ftr, chunksize=500000):
    grouped_frq_chunk_ftr = (
        frq_chunk_ftr
        .groupby(['AMOC', 'member', 'lat', 'lon', 'year'])['annual_frequency']
        .first()
        .reset_index()
    )
    frq_chunks_ftr.append(grouped_frq_chunk_ftr)

# This code reads a large CSV file for both HIST/FTR datasets in manageable chunks of 
# 500,000 rows to avoid exhausting system memory. 
# For each chunk, the data are grouped by AMOC state, ensemble member, latitude, longitude, and year, 
# the first annual frequency value associated with each unique combination is retained. 
# The resulting aggregated subset is then prepared for storage in a list, 
# allowing all chunks to be combined later for subsequent analysis. 

In [ ]:
df_frq_yearly_hist = (
    pd.concat(frq_chunks_hist)
    .groupby(['AMOC', 'member', 'lat', 'lon', 'year'])['annual_frequency']
    .first()
    .reset_index()
)

df_frq_yearly_ftr = (
    pd.concat(frq_chunks_ftr)
    .groupby(['AMOC', 'member', 'lat', 'lon', 'year'])['annual_frequency']
    .first()
    .reset_index()
)

# This code combines all previously processed historical data chunks into a single DataFrame 
# and groups the records by AMOC state, ensemble member, latitude, longitude, and year. 
# It aims to get rid of duplicate rows. 

In [ ]:
all_years_hist = list(range(1850,2014+1))
# all_years_ftr = list(range(2015,2100+1))

# Create list of years to restructure the years in the frq dataset.
# Need ot account for years without a thristwave.
# Could allow us to get % of years without thristwave. 

# Historical Calculation workflow -------------------------------------------------
# This code pivots the index into an excel spreadsheet and fills in years that have no value for frq. 
df_years_reindex_hist = (
    df_frq_yearly_hist
    .pivot(
        index=['AMOC', 'member', 'lat', 'lon'],
        columns='year',
        values='annual_frequency'
    )
    .fillna(0)
    .reindex(
        columns=all_years_hist, 
        fill_value=0
    )
)

# Mean across years -> one value per member/gridpoint
# creates an index of one value per member per gridpoint
hist_member_means_frq = ( # <-------------------------------------- Important for Histograms and showing tails of it
    df_years_reindex_hist
    .mean(axis=1)
    .reset_index(name='mean_annual_frequency')
)

# Mean across all 80 members
# Created an ensemble avg across members for each gridpoint. 
hist_ensemble_mean_frq = (
    hist_member_means_frq
    .groupby(['lat', 'lon'])['mean_annual_frequency']
    .mean()
    .reset_index()
)

# # Future Workflow ---------------------------------------------------------------------
# df_years_reindex_ftr = (
#     df_frq_yearly_ftr
#     .pivot(
#         index=['AMOC', 'member', 'lat', 'lon'],
#         columns='year',
#         values='annual_frequency'
#     )
#     .fillna(0)
#     .reindex(
#         columns=all_years_ftr, 
#         fill_value=0
#     )
# )

# ftr_member_means_frq = (
#     df_years_reindex_ftr
#     .mean(axis=1)
#     .reset_index(name='mean_annual_frequency')
# )

# ftr_ensemble_mean_frq = (
#     ftr_member_means_frq
#     .groupby(['lat', 'lon'])['mean_annual_frequency']
#     .mean()
#     .reset_index()
# )

In [ ]:
# Start with the hisotrical avg plot

# Create avg plot for JUST the first 22 years

In [ ]:
first_22 = list(range(2015,2036+1))
print(first_22)
print(len(first_22))

df_years_reindex_1 = (
    df_frq_yearly_ftr
    .pivot(
        index=['AMOC', 'member', 'lat', 'lon'],
        columns='year',
        values='annual_frequency'
    )
    .fillna(0)
    .reindex(
        columns=first_22, 
        fill_value=0
    )
)

ftr1_member_means_frq = (
    df_years_reindex_1
    .mean(axis=1)
    .reset_index(name='mean_annual_frequency')
)

ftr1_ensemble_mean_frq = (
    ftr1_member_means_frq
    .groupby(['lat', 'lon'])['mean_annual_frequency']
    .mean()
    .reset_index()
)

ftr1_ensemble_mean_frq

In [ ]:
df_years_reindex_1

In [ ]:
ftr1_member_means_frq

In [ ]:
second_21 = list(range(2037, 2058+1))
print(second_21)
print(len(second_21))

df_years_reindex_2 = (
    df_frq_yearly_ftr
    .pivot(
        index=['AMOC', 'member', 'lat', 'lon'],
        columns='year',
        values='annual_frequency'
    )
    .fillna(0)
    .reindex(
        columns=second_21, 
        fill_value=0
    )
)

ftr2_member_means_frq = (
    df_years_reindex_2
    .mean(axis=1)
    .reset_index(name='mean_annual_frequency')
)

ftr2_ensemble_mean_frq = (
    ftr2_member_means_frq
    .groupby(['lat', 'lon'])['mean_annual_frequency']
    .mean()
    .reset_index()
)

ftr2_ensemble_mean_frq

In [ ]:
third_21 = list(range(2059, 2079+1))
print(third_21)
print(len(third_21))

df_years_reindex_3 = (
    df_frq_yearly_ftr
    .pivot(
        index=['AMOC', 'member', 'lat', 'lon'],
        columns='year',
        values='annual_frequency'
    )
    .fillna(0)
    .reindex(
        columns=third_21, 
        fill_value=0
    )
)

ftr3_member_means_frq = (
    df_years_reindex_3
    .mean(axis=1)
    .reset_index(name='mean_annual_frequency')
)

ftr3_ensemble_mean_frq = (
    ftr3_member_means_frq
    .groupby(['lat', 'lon'])['mean_annual_frequency']
    .mean()
    .reset_index()
)

ftr3_ensemble_mean_frq

In [ ]:
fourth_21 = list(range(2080, 2100+1))
print(fourth_21)
print(len(fourth_21))

df_years_reindex_4 = (
    df_frq_yearly_ftr
    .pivot(
        index=['AMOC', 'member', 'lat', 'lon'],
        columns='year',
        values='annual_frequency'
    )
    .fillna(0)
    .reindex(
        columns=fourth_21, 
        fill_value=0
    )
)

ftr4_member_means_frq = (
    df_years_reindex_4
    .mean(axis=1)
    .reset_index(name='mean_annual_frequency')
)

ftr4_ensemble_mean_frq = (
    ftr4_member_means_frq
    .groupby(['lat', 'lon'])['mean_annual_frequency']
    .mean()
    .reset_index()
)

ftr4_ensemble_mean_frq

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import scipy.stats as stats
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D

# -------------------------------------------------------------------------
# Helper Function 0: Custom Symmetric Tick & Range Generator
# -------------------------------------------------------------------------
def generate_custom_ticks_1(grid_values, step=1):
    """
    Computes exact data min/max within CONUS, symmetric limits for pcolormesh,
    and ticks spaced by the step size, excluding the maximum value from 
    the tick marks while ensuring 0.0 is included.
    """
    exact_min = float(np.nanmin(grid_values))
    exact_max = float(np.nanmax(grid_values))
    
    if np.isnan(exact_min): 
        exact_min, exact_max = -1.0, 1.0
        
    # Symmetric limits rounded up to the nearest multiple of 'step'
    max_abs = np.ceil(max(abs(exact_min), abs(exact_max)) / step) * step
    if max_abs == 0:
        max_abs = step
        
    exact_min_sym = -max_abs
    exact_max_sym = max_abs
    
    # Generate potential ticks based on step increment
    num_steps = int(np.round((exact_max_sym - exact_min_sym) / step)) + 1
    
    # FIXED: Passed start, stop, and num_steps to np.linspace
    all_possible_ticks = np.round(np.linspace(exact_min_sym, exact_max_sym, num_steps), 2)
    
    # Filter ticks strictly within [exact_min, exact_max) - excluding exact_max
    custom_ticks = [t for t in all_possible_ticks if exact_min <= t < exact_max]
    
    # Explicitly include 0.0 if within bounds and not already present
    if exact_min <= 0.0 < exact_max:
        if not any(np.isclose(0.0, t, atol=1e-4) for t in custom_ticks):
            custom_ticks.append(0.0)
                
    # Sort ticks and clean up float precision
    custom_ticks = sorted(list(np.round(custom_ticks, 2)))
    tick_labels = [f"{t:.2f}" for t in custom_ticks]
    
    return custom_ticks, tick_labels, exact_max_sym, exact_min_sym, exact_min, exact_max

# -------------------------------------------------------------------------
# Helper Function 1: Compute Delta, T-Test, and CONUS Masking
# -------------------------------------------------------------------------
def process_period_delta(df_earlier, df_later, mask_3d):
    """
    Computes delta (later - earlier) and Welch's t-test per grid cell,
    then masks results to CONUS.
    """
    merged = pd.merge(
        df_earlier[['lat', 'lon', 'member', 'mean_annual_frequency']],
        df_later[['lat', 'lon', 'member', 'mean_annual_frequency']],
        on=['lat', 'lon', 'member'],
        suffixes=('_earlier', '_later')
    )

    results = []
    for (lat, lon), group in merged.groupby(['lat', 'lon']):
        earlier_vals = group['mean_annual_frequency_earlier'].values
        later_vals = group['mean_annual_frequency_later'].values

        _, p_val = stats.ttest_ind(
            later_vals,
            earlier_vals,
            equal_var=False,  # Welch's t-test
            alternative='greater',
            nan_policy='omit'
        )

        delta_val = np.nanmean(later_vals) - np.nanmean(earlier_vals)
        results.append([lat, lon, p_val, delta_val])

    results_df = pd.DataFrame(results, columns=['lat', 'lon', 'p_value', 'delta'])

    # Pivot to spatial 2D grids
    delta_grid = results_df.pivot(index='lat', columns='lon', values='delta')
    p_val_grid = results_df.pivot(index='lat', columns='lon', values='p_value')

    # Apply CONUS Mask
    delta_conus = delta_grid.where(mask_3d.values)
    p_val_conus = p_val_grid.where(mask_3d.values)

    return delta_conus, p_val_conus

# -------------------------------------------------------------------------
# Helper Function 2: Plotting Routine
# -------------------------------------------------------------------------
def plot_conus_delta_map(grid_delta, grid_pval, save_name, global_delta_values=None, alpha_sig=0.05):
    """
    Plots the delta grid as pcolormesh, calculates the percentage of CONUS grid cells
    with significant p-values, adds it to the title, and overlays a scatter layer.
    """
    values_for_ticks = global_delta_values if global_delta_values is not None else grid_delta.values
    
    custom_ticks, tick_labels, exact_max_sym, exact_min_sym, exact_min, exact_max = generate_custom_ticks_1(values_for_ticks)

    X, Y = np.meshgrid(grid_delta.columns, grid_delta.index)

    # Calculate Significant Lat/Lon Mask and Spatial Percentage
    pval_flat = grid_pval.stack().reset_index(name='p_value')
    valid_conus_points = pval_flat['p_value'].dropna()
    
    sig_mask = pval_flat['p_value'] < alpha_sig
    sig_lons = pval_flat.loc[sig_mask, 'lon'].values
    sig_lats = pval_flat.loc[sig_mask, 'lat'].values

    if len(valid_conus_points) > 0:
        sig_pct = (sig_mask.sum() / len(valid_conus_points)) * 100
    else:
        sig_pct = 0.0

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(18, 16))
    ax = plt.axes(projection=crs)
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())

    water_color = '#a5c9eb'
    ax.set_facecolor(water_color)

    # --- ZORDER 1: Base Data Grid ---
    im = ax.pcolormesh(
        X, Y, grid_delta.values,
        transform=ccrs.PlateCarree(),
        cmap='RdBu_r', 
        vmin=exact_min_sym, 
        vmax=exact_max_sym,
        shading='auto',
        zorder=1
    )

    # --- ZORDER 2: Scatter Layer for P-Values ---
    legend_label = f"o = p < {alpha_sig} ({sig_pct:.1f}% of CONUS)"
    if len(sig_lons) > 0:
        ax.scatter(
            sig_lons, sig_lats,
            transform=ccrs.PlateCarree(),
            s=50,
            marker='o', 
            facecolors='none', 
            edgecolors='black', 
            alpha=0.6,
            zorder=1
        )

    # --- ZORDER 3 & 4: Regional Geometries and Map Features ---
    if 'canada_geom' in globals() and canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), facecolor='#717376', edgecolor='black', linewidth=0.65, zorder=1)
    if 'mexico_geom' in globals() and mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), facecolor='#717376', edgecolor='black', linewidth=0.65, zorder=1)
    if 'great_lakes_geoms' in globals() and great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(), facecolor=water_color, edgecolor='black', linewidth=0.65, zorder=1)

    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=1)
    ax.coastlines('50m', linewidth=0.65, color='black', zorder=1)

    states = cfeature.NaturalEarthFeature(
        category='cultural', name='admin_1_states_provinces_lakes', scale='50m', facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.65, zorder=1)

    # --- ZORDER 5: Gridlines ---
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, color="#9d9d9d", zorder=1, alpha=0.8)
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True
    gl.x_inline = False
    gl.y_inline = False
    gl.xpadding = 4
    gl.ypadding = 4
    gl.xlabel_style = {'size': 20, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 20, 'color': 'black'}
    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))

# --- ZORDER 6: Top Level UI Overlay Boxes ---
    # 1. P-value Text Box Legend (Invisible handle removes extra marker symbol)
    if len(sig_lons) > 0:
        proxy_handle = Line2D(
            [0], [0],
            color='none',
            linestyle='none',
            marker='none',
            label=legend_label
        )

        leg = ax.legend(
            handles=[proxy_handle],
            loc='upper right', 
            frameon=True, 
            facecolor='white', 
            framealpha=0.9, 
            edgecolor='black',
            fontsize=20,
            handlelength=0,
            handletextpad=0
        )
        leg.set_zorder(2)

    # 2. Colorbar Inset Box Setup
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.55], zorder=2)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    cax = ax_box.inset_axes([0.10, 0.04, 0.39, 0.84])
    
    # Create Colorbar
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks)
    cb.ax.set_ylim(exact_min, exact_max)
    cb.ax.set_yticklabels(tick_labels, fontsize=18, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    ax_box.text(
        0.5, 0.94,
        "Δ Events",
        transform=ax_box.transAxes,
        fontsize=20,
        weight='bold', 
        color='black',
        ha='center',
        va='center'
    )

    plt.tight_layout()
    plt.savefig(f"{save_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

# =========================================================================
# Execution Workflow Across Time Transitions
# =========================================================================

# 1. Build CONUS Spatial Mask
sample_lats = hist_member_means_frq['lat'].unique()
sample_lons = hist_member_means_frq['lon'].unique()
frac_mask_3d = us_states.mask_3D_frac_approx(sample_lons, sample_lats, wrap_lon=True)
any_overlap_mask = (frac_mask_3d > 0).any(dim="region")

# 2. Define Period Transitions relative to HIST Baseline
comparisons = [
    ("(2015–2036) - HIST", hist_member_means_frq, ftr1_member_means_frq, "Delta_Hist_to_Ftr1"),
    ("(2037–2058) - HIST", hist_member_means_frq, ftr2_member_means_frq, "Delta_Hist_to_Ftr2"),
    ("(2059–2079) - HIST", hist_member_means_frq, ftr3_member_means_frq, "Delta_Hist_to_Ftr3"),
    ("(2080–2100) - HIST", hist_member_means_frq, ftr4_member_means_frq, "Delta_Hist_to_Ftr4")
]

# 3. Compute Delta and Statistical Grids
processed_results = []
for title_suffix, df_base, df_comp, save_name in comparisons:
    delta_grid, pval_grid = process_period_delta(df_base, df_comp, any_overlap_mask)
    
    processed_results.append({
        'title_prefix': "Thirstwave Frequency Delta | 80-Member Ensemble Mean",
        'title_suffix': title_suffix,
        'delta_grid': delta_grid,
        'pval_grid': pval_grid,
        'save_name': save_name
    })

# 4. Collect All CONUS Delta Values to Ensure Consistent Scaling Across Maps
all_delta_vals = np.concatenate([r['delta_grid'].values.flatten() for r in processed_results])

# 5. Render All Maps
for res in processed_results:
    plot_conus_delta_map(
        grid_delta=res['delta_grid'],
        grid_pval=res['pval_grid'],
        save_name=res['save_name'],
        global_delta_values=all_delta_vals,
        alpha_sig=0.05
    )